# ex01_lti_four
Interactive notebook version of the Python example.
Edit parameters and rerun cells to explore the model behavior.

## MATLAB Example Structure
- The fourth-order LTI model with coloured process noise
- Open-loop identification experiment
- Identification of the model in open loop
- Verification results
- Identification results
- Closed-loop identification experiment
- Identification of the model in closed loop
- Verification results
- Identification results

## Imports

In [ ]:
from __future__ import annotations
import numpy as np
from _common import (
    estimate_varmax_abcdk,
    estimate_varx_abcdk,
    format_metric,
    print_case_summary,
    simulate_closed_loop,
    vaf_percent,
)
from control import matlab as ml
from snr import snr_db

## The Fourth-Order LTI Model With Coloured Process Noise
State-space definition and the MATLAB-style open-loop and closed-loop systems.

In [ ]:
# The fourth-order LTI model with coloured process noise
a = np.array(
    [
        [0.67, 0.67, 0.0, 0.0],
        [-0.67, 0.67, 0.0, 0.0],
        [0.0, 0.0, -0.67, -0.67],
        [0.0, 0.0, 0.67, -0.67],
    ],
    dtype=np.float64,
)
b = np.array(
    [[0.6598, -0.5256], [1.9698, 0.4845], [4.3171, -0.4879], [-2.6436, -0.3416]],
    dtype=np.float64,
)
k = np.array(
    [[-0.6968, -0.1474], [0.1722, 0.5646], [0.6484, -0.4660], [-0.94, 0.1032]],
    dtype=np.float64,
)
c = np.array(
    [[-0.3749, 0.0751, -0.5225, 0.583], [-0.8977, 0.7543, 0.1159, 0.0982]],
    dtype=np.float64,
)
d = np.zeros((2, 2), dtype=np.float64)
f_gain = np.diag([0.25, 0.25]).astype(np.float64)

## Open-Loop Identification Experiment
Simulation of the model in open loop, matching the MATLAB `%% Open-loop identification experiment` section.

In [ ]:
# Simulation of the model in open loop
n_samples = 4000
t = np.arange(n_samples, dtype=np.float64)
rng = np.random.default_rng(101)
r = rng.standard_normal((n_samples, 2))
e = rng.standard_normal((n_samples, 2))
ol = ml.ss(a, np.hstack((b, k)), c, np.hstack((d, np.eye(c.shape[0]))), 1.0)
ol_nom = ml.ss(a, b, c, d, 1.0)
y, _, _ = ml.lsim(ol, np.hstack((r, e)), t)
y0, _, _ = ml.lsim(ol_nom, r, t)
y = np.asarray(y, dtype=np.float64)
y0 = np.asarray(y0, dtype=np.float64)
print("Signal to noise ratio (SNR) (open-loop)")
print(f"{format_metric(snr_db(y, y0))} dB")

Signal to noise ratio (SNR) (open-loop)
[11.31  7.35] dB


## Identification Of The Model In Open Loop
Order selection and model estimation for the open-loop data.

In [ ]:
# Parameters
n = 4
f = 10
p = 10
# PBSID-varx
_, ai, bi, ci, di, _, _, _ = estimate_varx_abcdk(r, y, n, f, p)
# PBSID-varmax
varmax_ok = True
try:
    _, av, bv, cv, dv, _ = estimate_varmax_abcdk(r, y, n, f, p)
except (np.linalg.LinAlgError, ValueError, FloatingPointError) as exc:
    varmax_ok = False
    print(f"[ex01-open-varmax] skipped due to numerical issue: {exc}")

## Verification Results
Open-loop verification against the identified models, mirroring MATLAB's `%% Verification results` section.

In [ ]:
yi, _, _ = ml.lsim(ml.ss(ai, bi, ci, di, 1.0), r, t)
yi = np.asarray(yi, dtype=np.float64)
if varmax_ok:
    yv, _, _ = ml.lsim(ml.ss(av, bv, cv, dv, 1.0), r, t)
    yv = np.asarray(yv, dtype=np.float64)
print_case_summary("ex01-open-varx", snr_db(y, y0), vaf_percent(y0, yi), ai)
if varmax_ok:
    print_case_summary("ex01-open-varmax", snr_db(y, y0), vaf_percent(y0, yv), av)


[ex01-open-varx]
SNR (dB): [11.31  7.35]
VAF (%): [99.87 99.57]
Estimated poles: [-0.6709+0.6705j -0.6709-0.6705j  0.6689+0.6735j  0.6689-0.6735j]

[ex01-open-varmax]
SNR (dB): [11.31  7.35]
VAF (%): [99.91 99.8 ]
Estimated poles: [-0.6682+0.6688j -0.6682-0.6688j  0.6676+0.6698j  0.6676-0.6698j]


## Identification Results
MATLAB plots pole locations and Bode magnitude here. The notebook keeps this section explicit and reports the identified models numerically.

## Closed-Loop Identification Experiment
Simulation of the model in closed loop, matching MATLAB's `%% Closed-loop identification experiment` section.

In [ ]:
u_cl, y_cl, y0_cl = simulate_closed_loop(a, b, c, d, k, f_gain, r, e=0.7 * e)
print("Signal to noise ratio (SNR) (closed-loop)")
print(f"{format_metric(snr_db(y_cl, y0_cl))} dB")

Signal to noise ratio (SNR) (closed-loop)
[9.9  9.38] dB


## Identification Of The Model In Closed Loop
Order selection and model estimation for the closed-loop data.

In [ ]:
_, ai_cl, bi_cl, ci_cl, di_cl, _, _, _ = estimate_varx_abcdk(u_cl, y_cl, n, f, p)
varmax_cl_ok = True
try:
    _, av_cl, bv_cl, cv_cl, dv_cl, _ = estimate_varmax_abcdk(u_cl, y_cl, n, f, p)
except (np.linalg.LinAlgError, ValueError, FloatingPointError) as exc:
    varmax_cl_ok = False
    print(f"[ex01-closed-varmax] skipped due to numerical issue: {exc}")

## Verification Results
Closed-loop verification against the identified models, mirroring MATLAB's second `%% Verification results` section.

In [ ]:
yi_cl, _, _ = ml.lsim(ml.ss(ai_cl, bi_cl, ci_cl, di_cl, 1.0), u_cl, t)
yi_cl = np.asarray(yi_cl, dtype=np.float64)
if varmax_cl_ok:
    yv_cl, _, _ = ml.lsim(ml.ss(av_cl, bv_cl, cv_cl, dv_cl, 1.0), u_cl, t)
    yv_cl = np.asarray(yv_cl, dtype=np.float64)
print_case_summary(
    "ex01-closed-varx", snr_db(y_cl, y0_cl), vaf_percent(y0_cl, yi_cl), ai_cl
)
if varmax_cl_ok:
    print_case_summary(
        "ex01-closed-varmax",
        snr_db(y_cl, y0_cl),
        vaf_percent(y0_cl, yv_cl),
        av_cl,
    )


[ex01-closed-varx]
SNR (dB): [9.9  9.38]
VAF (%): [98.86 99.22]
Estimated poles: [ 0.6555+0.6703j  0.6555-0.6703j -0.6482+0.6557j -0.6482-0.6557j]

[ex01-closed-varmax]
SNR (dB): [9.9  9.38]
VAF (%): [99.13 99.4 ]
Estimated poles: [-0.6505+0.6648j -0.6505-0.6648j  0.6593+0.6716j  0.6593-0.6716j]


## Identification Results
MATLAB ends with pole and Bode visualizations for the closed-loop case. The notebook keeps the section boundary explicit while the numerical summaries above capture the identified models.